# Object Detection in Images
### BCA Minor Project

This notebook demonstrates the same object-detection workflow used in the Streamlit project, with an additional **Top 3 Classification** column for each detected object.

The main detector remains **YOLO11n**. Detection is improved with a larger inference image size and test-time augmentation. A lightweight **YOLO11n classification model** is also used on each detected crop to provide up to three supplementary classification predictions.

> **Important:** The Top 3 Classification values are supplementary predictions from the classification model. They are not model-accuracy scores. The project still does **not** calculate accuracy or mAP.

## 1. Technology Stack
- Python
- Ultralytics YOLO11 (`yolo11n.pt`) for object detection
- Ultralytics YOLO11 classification (`yolo11n-cls.pt`) for Top 3 crop classification
- OpenCV
- NumPy
- Pandas
- Pillow
- Streamlit (used in the main application)

In [ ]:
# Install dependencies if required
# Run this cell once in a new environment.
!pip install ultralytics opencv-python numpy pandas pillow matplotlib

In [ ]:
import io
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
from PIL import Image
from ultralytics import YOLO

import matplotlib.pyplot as plt

## 2. Load the YOLO11 Models

Two pretrained models are used:

1. **YOLO11n detection model** — finds objects and bounding boxes.
2. **YOLO11n classification model** — examines each detected crop and returns up to three likely class labels.

The detection model uses a larger `imgsz` and test-time augmentation (`augment=True`) to help with difficult images. This can improve recall in some cases, although it cannot guarantee that every object will be classified correctly.

In [ ]:
DETECTION_MODEL_NAME = "yolo11n.pt"
CLASSIFICATION_MODEL_NAME = "yolo11n-cls.pt"

detection_model = YOLO(DETECTION_MODEL_NAME)
classification_model = YOLO(CLASSIFICATION_MODEL_NAME)

print(f"Loaded detection model: {DETECTION_MODEL_NAME}")
print(f"Loaded classification model: {CLASSIFICATION_MODEL_NAME}")

## 3. Upload / Select an Image

Set `IMAGE_PATH` to the image you want to test. The same image can be used in the Streamlit application.

In [ ]:
IMAGE_PATH = "test_image.jpg"  # Change this to your image path
image = Image.open(IMAGE_PATH).convert("RGB")
print("Image size:", image.size)
display(image)

## 4. Set Detection and Classification Settings

The confidence threshold filters object detections. It is **not an accuracy value**.

The classification model returns its top three class predictions for each detected crop.

In [ ]:
CONFIDENCE_THRESHOLD = 0.25
print("Detection confidence threshold:", CONFIDENCE_THRESHOLD)

## 5. Run Improved Object Detection

The detector uses:
- `imgsz=960` to give the model more image detail.
- `augment=True` for test-time augmentation.
- The selected confidence threshold.

These settings can help with small or difficult objects, but performance still depends on the image and the classes supported by the pretrained COCO model.

In [ ]:
rgb = np.array(image.convert("RGB"))
results = detection_model.predict(
    source=rgb,
    conf=CONFIDENCE_THRESHOLD,
    imgsz=960,
    augment=True,
    verbose=False
)
result = results[0]

print(f"Detected {len(result.boxes) if result.boxes is not None else 0} object(s).")

## 6. Display Detected Image

Bounding boxes and labels are drawn using the YOLO detection result.

In [ ]:
annotated = Image.fromarray(cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB))
display(annotated)

## 7. Extract Detection Details + Top 3 Classification

For every detected bounding box, the table includes:
- detected object name
- detection confidence
- bounding-box coordinates
- **Top 3 Classification** from the classification model applied to that object crop

The classification output is supplementary because the detector and classifier use different pretrained label sets. A classification model can also make mistakes, especially for very small or unusual crops.

In [ ]:
def get_top3_classifications(crop):
    """Return up to three classification labels with probabilities for one object crop."""
    try:
        cls_results = classification_model.predict(source=np.array(crop), verbose=False)
        cls_result = cls_results[0]
        if cls_result.probs is None:
            return "Not available"

        top_indices = cls_result.probs.top5[:3]
        top_values = cls_result.probs.top5conf[:3]
        names = cls_result.names

        predictions = []
        for idx, prob in zip(top_indices, top_values):
            label = names[int(idx)]
            predictions.append(f"{label} ({float(prob) * 100:.1f}%)")
        return " | ".join(predictions)
    except Exception as e:
        return f"Not available: {type(e).__name__}"

rows = []
if result.boxes is not None:
    boxes = result.boxes.xyxy.cpu().numpy()
    confs = result.boxes.conf.cpu().numpy()
    classes = result.boxes.cls.cpu().numpy().astype(int)

    image_width, image_height = image.size

    for box, conf, cls_id in zip(boxes, confs, classes):
        x1, y1, x2, y2 = box
        # Keep crop coordinates inside the image.
        x1i = max(0, min(int(x1), image_width - 1))
        y1i = max(0, min(int(y1), image_height - 1))
        x2i = max(x1i + 1, min(int(x2), image_width))
        y2i = max(y1i + 1, min(int(y2), image_height))

        crop = image.crop((x1i, y1i, x2i, y2i))
        top3 = get_top3_classifications(crop)

        rows.append({
            "Object": result.names[int(cls_id)],
            "Confidence": round(float(conf) * 100, 2),
            "Top 3 Classification": top3,
            "X1": round(float(x1), 1),
            "Y1": round(float(y1), 1),
            "X2": round(float(x2), 1),
            "Y2": round(float(y2), 1)
        })

df = pd.DataFrame(
    rows,
    columns=["Object", "Confidence", "Top 3 Classification", "X1", "Y1", "X2", "Y2"]
)

df

## 8. Project Metrics

These are descriptive detection outputs, **not accuracy metrics**.

In [ ]:
if df.empty:
    print("No objects were detected above the selected threshold.")
else:
    print("Objects Detected:", len(df))
    print("Unique Classes:", df["Object"].nunique())
    print("Average Detection Confidence: {:.2f}%".format(df["Confidence"].mean()))

## 9. Save the Detection Result
The Streamlit app saves its latest result in `output_images/latest_detection.jpg`.

In [ ]:
OUTPUT_DIR = Path("output_images")
OUTPUT_DIR.mkdir(exist_ok=True)
output_path = OUTPUT_DIR / "latest_detection.jpg"
annotated.save(output_path, quality=95)
print("Saved result to:", output_path)

## 10. Download/Export in Memory
This reproduces the image-export logic used by the Streamlit application.

In [ ]:
buffer = io.BytesIO()
annotated.save(buffer, format="JPEG", quality=95)
print("JPEG output created in memory.")

## 11. Working Flow

**Input Image → Preprocessing → YOLO11 Detection → Bounding Boxes → Object Names & Confidence → Crop Each Detected Object → Top 3 Classification → Visualization**

### Why some objects may still be missed
- The pretrained detector is trained on a fixed set of common object classes.
- Very small, blurry, hidden, or unusual objects may be missed.
- Lowering the detection threshold can increase detections but may also introduce false positives.
- Test-time augmentation and a larger image size can help, but they do not guarantee perfect detection.
- For a specific custom object category, a custom labelled dataset and fine-tuning are the appropriate next step.

## 12. Conclusion

The notebook demonstrates an image-based object detection system using YOLO11 and adds a **Top 3 Classification** column for each detected object. The improved inference settings are intended to help with difficult images while keeping the project lightweight.

**Evaluation note:** This project does **not** calculate accuracy or mAP. Confidence scores and Top 3 Classification probabilities are model outputs, not accuracy measurements. Proper accuracy/mAP evaluation requires labelled ground-truth test data and a separate evaluation pipeline.